# Ensemble Scenarios

Three ensemble strategies: A (majority vote), B (weighted average), C (meta-learner).

## Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    roc_auc_score, accuracy_score, f1_score,
    roc_curve, confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.linear_model import LogisticRegression

warnings.filterwarnings('ignore')
matplotlib.rcParams['figure.dpi'] = 100

# ── Chemins ──────────────────────────────────────────────────────
BASE        = '/content/drive/MyDrive/Memoire_Deepfakes'
RESULTS_DIR = f'{BASE}/data/results'
SPLITS_DIR  = f'{BASE}/data/splits'
FIGS_DIR    = f'{RESULTS_DIR}/figures'
os.makedirs(FIGS_DIR, exist_ok=True)

# ── Colonnes (issues de NB04) ─────────────────────────────────────
PROB_COLS   = ['P_Meso4', 'P_XceptionNet', 'P_UCF', 'P_F3Net']
MODEL_NAMES = ['Meso4',   'XceptionNet',   'UCF',   'F3Net']
LABEL_COL   = 'label'   # 0 = REAL, 1 = FAKE

# ── Chargement ───────────────────────────────────────────────────
train_df = pd.read_csv(f'{RESULTS_DIR}/train_probs.csv')
test_df  = pd.read_csv(f'{RESULTS_DIR}/test_probs.csv')

# ── Vérifications ────────────────────────────────────────────────
print('=' * 65)
print('CHARGEMENT DES DONNÉES')
print('=' * 65)
EXPECTED = {'train': 1402, 'test': 2807}
for split_name, df in [('train', train_df), ('test', test_df)]:
    exp = EXPECTED[split_name]
    ok  = len(df) == exp
    print(f'\n  {split_name}_probs.csv :')
    print(f'    Lignes      : {len(df):,} / {exp:,} attendues  {"✅" if ok else "⚠️"}')
    print(f'    Colonnes    : {list(df.columns)}')
    dist = df[LABEL_COL].value_counts().sort_index().to_dict()
    print(f'    Labels      : {dist}  (0=REAL, 1=FAKE)')
    for col in PROB_COLS:
        if col not in df.columns:
            raise KeyError(f'⚠️  Colonne absente : {col} — vérifier NB04')
        n_nan = int(df[col].isna().sum())
        flag  = '✅' if n_nan == 0 else f'⚠️  {n_nan} NaN'
        print(f'    {flag}  {col:<16s}: [{df[col].min():.4f}, {df[col].max():.4f}]')

# ── Garde-fou coffre-fort ─────────────────────────────────────────
for forbidden in ['val_probs.csv', 'ensemble_scores_val.csv']:
    if os.path.isfile(f'{RESULTS_DIR}/{forbidden}'):
        raise RuntimeError(f'⛔  {forbidden} détecté — COFFRE-FORT COMPROMIS. Arrêt immédiat.')

print()
print('  ✅ Coffre-fort val : INTACT')
print('=' * 65)

# Vecteurs labels
y_train = train_df[LABEL_COL].values
y_test  = test_df[LABEL_COL].values

# Stockage centralisé des résultats
RESULTS = {}


## Metrics Functions

In [ ]:
def compute_eer(y_true, y_scores):
    """
    Equal Error Rate : seuil où FPR ≈ FNR.
    Retourne la moyenne interpolée (fpr[idx] + fnr[idx]) / 2.
    """
    fpr, tpr, _ = roc_curve(y_true, y_scores)
    fnr = 1.0 - tpr
    idx = int(np.nanargmin(np.abs(fpr - fnr)))
    return float((fpr[idx] + fnr[idx]) / 2.0)


def compute_all_metrics(y_true, y_scores, threshold=0.5):
    """Retourne {AUC, Accuracy, F1, EER} pour un jeu de scores."""
    y_pred = (np.asarray(y_scores) >= threshold).astype(int)
    return {
        'AUC'      : float(roc_auc_score(y_true, y_scores)),
        'Accuracy' : float(accuracy_score(y_true, y_pred)),
        'F1'       : float(f1_score(y_true, y_pred, zero_division=0)),
        'EER'      : compute_eer(y_true, y_scores),
    }


def print_metrics_row(label, m_train, m_test):
    hdr = f'  {label}'
    print(hdr)
    print(f'    Train → AUC={m_train["AUC"]:.4f}  '
          f'Acc={m_train["Accuracy"]:.4f}  '
          f'F1={m_train["F1"]:.4f}  '
          f'EER={m_train["EER"]:.4f}')
    print(f'    Test  → AUC={m_test["AUC"]:.4f}  '
          f'Acc={m_test["Accuracy"]:.4f}  '
          f'F1={m_test["F1"]:.4f}  '
          f'EER={m_test["EER"]:.4f}')


print('✅ Fonctions utilitaires définies.')


## Individual Detector Metrics

In [ ]:
print('=' * 70)
print('MÉTRIQUES INDIVIDUELLES — 4 modèles (poids gelés FF++ c23)')
print('Dataset : diffusion (MidJourney + ddim + DiT + SiT) vs REAL')
print('=' * 70)

for model, col in zip(MODEL_NAMES, PROB_COLS):
    s_train = train_df[col].values
    s_test  = test_df[col].values

    m_train = compute_all_metrics(y_train, s_train)
    m_test  = compute_all_metrics(y_test,  s_test)

    RESULTS[model] = {
        'train'        : m_train,
        'test'         : m_test,
        'scores_train' : s_train,
        'scores_test'  : s_test,
    }

    print()
    print_metrics_row(model, m_train, m_test)

print()
print('=' * 70)
print('✅ Métriques individuelles calculées et stockées dans RESULTS.')


## Scenario A: Majority Vote

In [ ]:
print('=' * 70)
print('SCÉNARIO A — VOTE MAJORITAIRE')
print('  Règle de décision : n_votes_FAKE >= 2  →  FAKE (tie-break inclus)')
print('  Score continu AUC : moyenne des 4 probabilités P(FAKE)')
print('=' * 70)

for split_name, df, y_true in [
        ('train', train_df, y_train),
        ('test',  test_df,  y_test)]:

    # Votes binaires individuels
    votes        = (df[PROB_COLS].values >= 0.5).astype(int)  # (N, 4)
    n_fake_votes = votes.sum(axis=1)                           # 0..4

    # Score continu (pour AUC)
    score_A = df[PROB_COLS].mean(axis=1).values

    # Prédiction binaire (seuil ≥ 2 pour tie-break FAKE)
    pred_A = (n_fake_votes >= 2).astype(int)

    m = compute_all_metrics(y_true, score_A)
    # Override Accuracy et F1 avec prédictions binaires par vote
    m['Accuracy'] = float(accuracy_score(y_true, pred_A))
    m['F1']       = float(f1_score(y_true, pred_A, zero_division=0))

    # Distribution des votes
    dist = {v: int((n_fake_votes == v).sum()) for v in range(5)}

    if split_name == 'train':
        RESULTS['Scenario_A'] = {
            'train'        : m,
            'scores_train' : score_A,
            'preds_train'  : pred_A,
        }
    else:
        RESULTS['Scenario_A']['test']        = m
        RESULTS['Scenario_A']['scores_test'] = score_A
        RESULTS['Scenario_A']['preds_test']  = pred_A

    print(f'\n  [{split_name.upper()}]')
    print(f'    AUC={m["AUC"]:.4f}  Acc={m["Accuracy"]:.4f}  '
          f'F1={m["F1"]:.4f}  EER={m["EER"]:.4f}')
    print(f'    Distribution votes FAKE/image : {dist}')

print()
print('✅ Scénario A calculé et stocké dans RESULTS.')


## Scenario B: Weighted Average

In [ ]:
print('=' * 70)
print('SCÉNARIO B — MOYENNE PONDÉRÉE')
print('  Calibration des poids : accuracy individuelle sur TRAIN SET')
print('=' * 70)

# ── Calibration des poids sur le TRAIN SET ───────────────────────
raw_weights = {}
print('\n  Étape 1 — Accuracy par modèle sur le Train Set :')
for model, col in zip(MODEL_NAMES, PROB_COLS):
    pred_train = (train_df[col].values >= 0.5).astype(int)
    acc_train  = float(accuracy_score(y_train, pred_train))
    raw_weights[col] = acc_train
    print(f'    {model:<14s}  accuracy = {acc_train:.4f}')

total_w      = sum(raw_weights.values())
norm_weights = {col: w / total_w for col, w in raw_weights.items()}

print('\n  Étape 2 — Poids normalisés (somme = 1.0000) :')
for model, col in zip(MODEL_NAMES, PROB_COLS):
    print(f'    {model:<14s}  w = {norm_weights[col]:.4f}')

# ── Application Train + Test ──────────────────────────────────────
print('\n  Étape 3 — Application de la moyenne pondérée :')
for split_name, df, y_true in [
        ('train', train_df, y_train),
        ('test',  test_df,  y_test)]:

    score_B = sum(norm_weights[col] * df[col].values for col in PROB_COLS)
    pred_B  = (score_B >= 0.5).astype(int)

    m = compute_all_metrics(y_true, score_B)

    if split_name == 'train':
        RESULTS['Scenario_B'] = {
            'train'        : m,
            'scores_train' : score_B,
            'weights'      : norm_weights,
        }
    else:
        RESULTS['Scenario_B']['test']        = m
        RESULTS['Scenario_B']['scores_test'] = score_B

    print(f'\n  [{split_name.upper()}]')
    print(f'    AUC={m["AUC"]:.4f}  Acc={m["Accuracy"]:.4f}  '
          f'F1={m["F1"]:.4f}  EER={m["EER"]:.4f}')

print()
print('✅ Scénario B calculé et stocké dans RESULTS.')


In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.utils import resample
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# ============================================================================
# SECTION 1: CHARGEMENT DES DONNÉES
# ============================================================================
# À adapter selon ton architecture de fichiers

# Charger les probabilités du Training Set
train_probs = pd.read_csv('/content/drive/MyDrive/Memoire_Deepfakes/data/results/train_probs.csv')

# Filtrer uniquement le Training Set
train_data = train_probs[train_probs['split'] == 'train'].copy()

print(f"Taille du Training Set: {len(train_data)} images")
print(f"Distribution des labels: {train_data['label'].value_counts().to_dict()}")

# Préparer les features (probabilités des 4 détecteurs) et la cible
X_train = train_data[['P_Meso4', 'P_XceptionNet', 'P_UCF', 'P_F3Net']].values
y_train = train_data['label'].values

# ============================================================================
# SECTION 2: FONCTION DE BOOTSTRAP
# ============================================================================

def bootstrap_meta_learner(X, y, n_iterations=1000, random_state=42):
    """
    Effectue une analyse de sensibilité par bootstrap sur le méta-learner.

    Parameters:
    -----------
    X : array-like, shape (n_samples, n_features)
        Features d'entrée (probabilités des détecteurs)
    y : array-like, shape (n_samples,)
        Labels cibles
    n_iterations : int
        Nombre d'itérations bootstrap
    random_state : int
        Seed pour la reproductibilité

    Returns:
    --------
    coefficients : dict
        Dictionnaire contenant les coefficients pour chaque détecteur
    intercepts : array
        Intercepts pour chaque itération
    """

    np.random.seed(random_state)

    # Initialiser les conteneurs pour les coefficients
    coef_meso4 = []
    coef_xceptionnet = []
    coef_ucf = []
    coef_f3net = []
    intercepts = []

    # Itérations bootstrap
    for i in range(n_iterations):
        # Rééchantillonnage avec remplacement
        X_resampled, y_resampled = resample(X, y,
                                             n_samples=len(X),
                                             random_state=random_state + i,
                                             stratify=y)

        # Entraîner le méta-learner sur l'échantillon bootstrap
        # IMPORTANT: Utiliser penalty='l2', C=1.0 pour correspondre au notebook 05_ensemble.ipynb
        meta_learner = LogisticRegression(
            penalty='l2',
            C=1.0,
            max_iter=1000,
            random_state=random_state + i,
            solver='lbfgs'
        )
        meta_learner.fit(X_resampled, y_resampled)

        # Extraire les coefficients
        coef_meso4.append(meta_learner.coef_[0][0])
        coef_xceptionnet.append(meta_learner.coef_[0][1])
        coef_ucf.append(meta_learner.coef_[0][2])
        coef_f3net.append(meta_learner.coef_[0][3])
        intercepts.append(meta_learner.intercept_[0])

        # Afficher la progression tous les 100 itérations
        if (i + 1) % 100 == 0:
            print(f"Progression: {i + 1}/{n_iterations} itérations complétées")

    coefficients = {
        'Meso4': np.array(coef_meso4),
        'XceptionNet': np.array(coef_xceptionnet),
        'UCF': np.array(coef_ucf),
        'F3Net': np.array(coef_f3net)
    }

    return coefficients, np.array(intercepts)

# ============================================================================
# SECTION 3: EXÉCUTION DU BOOTSTRAP
# ============================================================================

print("\n" + "="*80)
print("ANALYSE DE SENSIBILITÉ PAR BOOTSTRAP - MÉTA-LEARNER (SCÉNARIO C)")
print("="*80 + "\n")

print("Lancement du bootstrap avec 1000 itérations...")
coefficients, intercepts = bootstrap_meta_learner(X_train, y_train,
                                                   n_iterations=1000,
                                                   random_state=42)

# ============================================================================
# SECTION 4: CALCUL DES STATISTIQUES
# ============================================================================

print("\n" + "-"*80)
print("RÉSULTATS DE L'ANALYSE DE SENSIBILITÉ")
print("-"*80 + "\n")

# Tableau récapitulatif
results = []
for detector_name, coef_values in coefficients.items():
    mean_coef = np.mean(coef_values)
    std_coef = np.std(coef_values)
    ci_lower = np.percentile(coef_values, 2.5)
    ci_upper = np.percentile(coef_values, 97.5)

    # Test si l'intervalle de confiance inclut zéro
    includes_zero = (ci_lower <= 0) and (ci_upper >= 0)

    results.append({
        'Détecteur': detector_name,
        'Moyenne_β': mean_coef,
        'Écart-type': std_coef,
        'IC_95%_Inf': ci_lower,
        'IC_95%_Sup': ci_upper,
        'Inclut_zéro': includes_zero
    })

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

# Statistiques pour l'intercept
print(f"\nIntercept:")
print(f"  Moyenne: {np.mean(intercepts):.4f}")
print(f"  Écart-type: {np.std(intercepts):.4f}")
print(f"  IC 95%: [{np.percentile(intercepts, 2.5):.4f}, {np.percentile(intercepts, 97.5):.4f}]")

# ============================================================================
# SECTION 5: ANALYSE SPÉCIFIQUE DU COEFFICIENT β_MESO4
# ============================================================================

print("\n" + "-"*80)
print("ANALYSE DÉTAILLÉE DU COEFFICIENT β_MESO4")
print("-"*80 + "\n")

beta_meso4 = coefficients['Meso4']

# Statistiques descriptives
print(f"Moyenne: {np.mean(beta_meso4):.4f}")
print(f"Médiane: {np.median(beta_meso4):.4f}")
print(f"Écart-type: {np.std(beta_meso4):.4f}")
print(f"Coefficient de variation: {(np.std(beta_meso4) / abs(np.mean(beta_meso4))) * 100:.2f}%")
print(f"Min: {np.min(beta_meso4):.4f}")
print(f"Max: {np.max(beta_meso4):.4f}")
print(f"IC 95%: [{np.percentile(beta_meso4, 2.5):.4f}, {np.percentile(beta_meso4, 97.5):.4f}]")

# Proportion de coefficients négatifs
prop_negative = (beta_meso4 < 0).sum() / len(beta_meso4) * 100
print(f"\nProportion d'itérations avec β < 0: {prop_negative:.2f}%")

# Test de significativité (t-test contre 0)
t_stat, p_value = stats.ttest_1samp(beta_meso4, 0)
print(f"\nTest t contre H₀: β = 0")
print(f"  t-statistic: {t_stat:.4f}")
print(f"  p-value: {p_value:.2e}")
print(f"  Significatif: {'Oui' if p_value < 0.05 else 'Non'}")

# ============================================================================
# SECTION 6: VISUALISATIONS
# ============================================================================

# Configuration du style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (16, 10)

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('Analyse de sensibilité par bootstrap - Coefficients du méta-learner (Scénario C)',
             fontsize=16, fontweight='bold')

# Subplot 1: Distribution de β_Meso4
ax1 = axes[0, 0]
ax1.hist(beta_meso4, bins=50, edgecolor='black', alpha=0.7, color='#e74c3c')
ax1.axvline(np.mean(beta_meso4), color='red', linestyle='--', linewidth=2, label=f'Moyenne: {np.mean(beta_meso4):.4f}')
ax1.axvline(np.percentile(beta_meso4, 2.5), color='orange', linestyle=':', linewidth=2, label=f'IC 95%: [{np.percentile(beta_meso4, 2.5):.4f}, {np.percentile(beta_meso4, 97.5):.4f}]')
ax1.axvline(np.percentile(beta_meso4, 97.5), color='orange', linestyle=':', linewidth=2)
ax1.axvline(0, color='black', linestyle='-', linewidth=1, alpha=0.5)
ax1.set_xlabel('Coefficient β_Meso4', fontweight='bold')
ax1.set_ylabel('Fréquence', fontweight='bold')
ax1.set_title('Distribution de β_Meso4 (1000 itérations)', fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Subplot 2: Distribution de β_XceptionNet
ax2 = axes[0, 1]
beta_xception = coefficients['XceptionNet']
ax2.hist(beta_xception, bins=50, edgecolor='black', alpha=0.7, color='#3498db')
ax2.axvline(np.mean(beta_xception), color='blue', linestyle='--', linewidth=2, label=f'Moyenne: {np.mean(beta_xception):.4f}')
ax2.axvline(0, color='black', linestyle='-', linewidth=1, alpha=0.5)
ax2.set_xlabel('Coefficient β_XceptionNet', fontweight='bold')
ax2.set_ylabel('Fréquence', fontweight='bold')
ax2.set_title('Distribution de β_XceptionNet', fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

# Subplot 3: Distribution de β_UCF
ax3 = axes[0, 2]
beta_ucf = coefficients['UCF']
ax3.hist(beta_ucf, bins=50, edgecolor='black', alpha=0.7, color='#2ecc71')
ax3.axvline(np.mean(beta_ucf), color='green', linestyle='--', linewidth=2, label=f'Moyenne: {np.mean(beta_ucf):.4f}')
ax3.axvline(0, color='black', linestyle='-', linewidth=1, alpha=0.5)
ax3.set_xlabel('Coefficient β_UCF', fontweight='bold')
ax3.set_ylabel('Fréquence', fontweight='bold')
ax3.set_title('Distribution de β_UCF', fontweight='bold')
ax3.legend()
ax3.grid(True, alpha=0.3)

# Subplot 4: Distribution de β_F3Net
ax4 = axes[1, 0]
beta_f3net = coefficients['F3Net']
ax4.hist(beta_f3net, bins=50, edgecolor='black', alpha=0.7, color='#9b59b6')
ax4.axvline(np.mean(beta_f3net), color='purple', linestyle='--', linewidth=2, label=f'Moyenne: {np.mean(beta_f3net):.4f}')
ax4.axvline(0, color='black', linestyle='-', linewidth=1, alpha=0.5)
ax4.set_xlabel('Coefficient β_F3Net', fontweight='bold')
ax4.set_ylabel('Fréquence', fontweight='bold')
ax4.set_title('Distribution de β_F3Net', fontweight='bold')
ax4.legend()
ax4.grid(True, alpha=0.3)

# Subplot 5: Boxplot comparatif des 4 coefficients
ax5 = axes[1, 1]
data_for_boxplot = [coefficients['Meso4'], coefficients['XceptionNet'],
                     coefficients['UCF'], coefficients['F3Net']]
bp = ax5.boxplot(data_for_boxplot, labels=['Meso4', 'XceptionNet', 'UCF', 'F3Net'],
                  patch_artist=True, showmeans=True)
colors = ['#e74c3c', '#3498db', '#2ecc71', '#9b59b6']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax5.axhline(0, color='black', linestyle='-', linewidth=1, alpha=0.5)
ax5.set_ylabel('Valeur du coefficient β', fontweight='bold')
ax5.set_title('Comparaison des distributions de coefficients', fontweight='bold')
ax5.grid(True, alpha=0.3, axis='y')

# Subplot 6: Q-Q plot pour β_Meso4 (test de normalité)
ax6 = axes[1, 2]
stats.probplot(beta_meso4, dist="norm", plot=ax6)
ax6.set_title('Q-Q Plot - β_Meso4 (Test de normalité)', fontweight='bold')
ax6.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/Memoire_Deepfakes/data/results/bootstrap_sensitivity_analysis.png',
            dpi=300, bbox_inches='tight')
print("\n✓ Visualisation sauvegardée: bootstrap_sensitivity_analysis.png")

# ============================================================================
# SECTION 7: SAUVEGARDE DES RÉSULTATS
# ============================================================================

# Sauvegarder le tableau récapitulatif
results_df.to_csv('/content/drive/MyDrive/Memoire_Deepfakes/data/results/bootstrap_coefficients_summary.csv',
                  index=False)
print("✓ Tableau récapitulatif sauvegardé: bootstrap_coefficients_summary.csv")

# Sauvegarder toutes les itérations pour analyses ultérieures
bootstrap_iterations = pd.DataFrame({
    'iteration': range(1000),
    'beta_Meso4': coefficients['Meso4'],
    'beta_XceptionNet': coefficients['XceptionNet'],
    'beta_UCF': coefficients['UCF'],
    'beta_F3Net': coefficients['F3Net'],
    'intercept': intercepts
})
bootstrap_iterations.to_csv('/content/drive/MyDrive/Memoire_Deepfakes/data/results/bootstrap_iterations.csv',
                            index=False)
print("✓ Détail des itérations sauvegardé: bootstrap_iterations.csv")

# ============================================================================
# SECTION 8: INTERPRÉTATION POUR LE MÉMOIRE
# ============================================================================

print("\n" + "="*80)
print("SYNTHÈSE POUR INTÉGRATION DANS LE MÉMOIRE")
print("="*80 + "\n")

print("L'analyse de sensibilité par bootstrap (1000 itérations) confirme la robustesse")
print("du coefficient β_Meso4 du méta-learner:")
print()
print(f"• Coefficient moyen: β_Meso4 = {np.mean(beta_meso4):.4f} (IC 95%: [{np.percentile(beta_meso4, 2.5):.4f}, {np.percentile(beta_meso4, 97.5):.4f}])")
print(f"• Écart-type: {np.std(beta_meso4):.4f}")
print(f"• {prop_negative:.1f}% des itérations produisent un coefficient négatif")
print(f"• p-value (test contre H₀: β = 0): {p_value:.2e} → fortement significatif")
print()
print("CONCLUSION: Le caractère fortement négatif de β_Meso4 n'est PAS un artefact")
print("échantillonnal. L'intervalle de confiance exclut complètement zéro, confirmant")
print("que le méta-learner inverse systématiquement le signal de Meso4 pour améliorer")
print("la détection. Cette stratégie d'inversion est robuste et reproductible.")

print("\n" + "="*80)
print("ANALYSE COMPLÉTÉE")
print("="*80)


## Scenario C: Meta-Learner

In [ ]:
print('=' * 70)
print('SCÉNARIO C — MÉTA-LEARNER (Régression Logistique)')
print('  Features  : P_Meso4, P_XceptionNet, P_UCF, P_F3Net  (4 colonnes)')
print('  Entraîné sur : TRAIN SET uniquement  (1 402 observations)')
print('  Évalué sur   : TRAIN SET + TEST SET  (hors-échantillon)')
print('=' * 70)

X_train = train_df[PROB_COLS].values   # (1402, 4)
X_test  = test_df[PROB_COLS].values    # (2807, 4)

# ── Entraînement ─────────────────────────────────────────────────
clf = LogisticRegression(max_iter=1000, random_state=42, solver='lbfgs')
clf.fit(X_train, y_train)

print('\n  Coefficients appris (β) :')
for model, coef in zip(MODEL_NAMES, clf.coef_[0]):
    sign = '+' if coef >= 0 else ''
    print(f'    {model:<14s}  β = {sign}{coef:.4f}')
print(f'    Intercept       β0 = {clf.intercept_[0]:+.4f}')

# ── Scores de probabilité méta ───────────────────────────────────
score_C_train = clf.predict_proba(X_train)[:, 1]   # P(FAKE) méta — Train
score_C_test  = clf.predict_proba(X_test)[:, 1]    # P(FAKE) méta — Test

# ── Métriques ────────────────────────────────────────────────────
print('\n  Performances :')
for split_name, y_true, score_C in [
        ('train', y_train, score_C_train),
        ('test',  y_test,  score_C_test)]:

    pred_C = (score_C >= 0.5).astype(int)
    m = {
        'AUC'      : float(roc_auc_score(y_true, score_C)),
        'Accuracy' : float(accuracy_score(y_true, pred_C)),
        'F1'       : float(f1_score(y_true, pred_C, zero_division=0)),
        'EER'      : compute_eer(y_true, score_C),
    }

    if split_name == 'train':
        RESULTS['Scenario_C'] = {
            'train'        : m,
            'scores_train' : score_C_train,
            'model'        : clf,
        }
    else:
        RESULTS['Scenario_C']['test']        = m
        RESULTS['Scenario_C']['scores_test'] = score_C_test

    print(f'\n  [{split_name.upper()}]')
    print(f'    AUC={m["AUC"]:.4f}  Acc={m["Accuracy"]:.4f}  '
          f'F1={m["F1"]:.4f}  EER={m["EER"]:.4f}')

# ── Contrôle anti-overfitting ─────────────────────────────────────
auc_tr  = RESULTS['Scenario_C']['train']['AUC']
auc_te  = RESULTS['Scenario_C']['test']['AUC']
auc_gap = abs(auc_tr - auc_te)

print()
print('  ━━━━ Contrôle anti-overfitting (seuil VDK : 10 pp) ━━━━')
print(f'    AUC Train  : {auc_tr:.4f}')
print(f'    AUC Test   : {auc_te:.4f}')
print(f'    Écart      : {auc_gap:.4f}  ({auc_gap*100:.2f} pp)')

if auc_gap > 0.10:
    print('    ⚠️  ALERTE OVERFITTING — Écart > 10 points de pourcentage')
    print('       → Recommandation : augmenter la régularisation (C < 1.0)')
    print('         ou ajouter penalty=\'l2\' avec C=0.1')
else:
    print(f'    ✅ Écart acceptable (< 10 pp) — pas de signe d\'overfitting')

print()
print('✅ Scénario C calculé et stocké dans RESULTS.')


## Visualizations

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression

# Charger les données
train = pd.read_csv('ensemble_scores_train.csv')
val = pd.read_csv('val_probs.csv')

# === ENTRAÎNER LE MÉTA-LEARNER ===
X_train = train[['P_Meso4', 'P_XceptionNet', 'P_UCF', 'P_F3Net']].values
y_train = train['label'].values

meta_model = LogisticRegression(penalty='l2', C=1.0, max_iter=1000, random_state=42)
meta_model.fit(X_train, y_train)

# Prédire sur validation
X_val = val[['P_Meso4', 'P_XceptionNet', 'P_UCF', 'P_F3Net']].values
val['score_C'] = meta_model.predict_proba(X_val)[:, 1]

# === CALCULER LES STATISTIQUES POUR LES IMAGES FAKE ===
val_fake = val[val['label'] == 1].copy()

# Scénario A : moyenne simple
val_fake['score_A'] = (val_fake['P_Meso4'] + val_fake['P_XceptionNet'] +
                        val_fake['P_UCF'] + val_fake['P_F3Net']) / 4

# Scénario B : moyenne pondérée par accuracy du train
w_meso4 = (train['label'] == (train['P_Meso4'] > 0.5)).mean()
w_xception = (train['label'] == (train['P_XceptionNet'] > 0.5)).mean()
w_ucf = (train['label'] == (train['P_UCF'] > 0.5)).mean()
w_f3net = (train['label'] == (train['P_F3Net'] > 0.5)).mean()
total_w = w_meso4 + w_xception + w_ucf + w_f3net

val_fake['score_B'] = (w_meso4 * val_fake['P_Meso4'] +
                        w_xception * val_fake['P_XceptionNet'] +
                        w_ucf * val_fake['P_UCF'] +
                        w_f3net * val_fake['P_F3Net']) / total_w

# === AFFICHER LES RÉSULTATS ===
print("=" * 70)
print("STATISTIQUES DES SCORES P(FAKE) - IMAGES FAKE UNIQUEMENT")
print("=" * 70)
print()
print("Modèle          P(FAKE) minimum   P(FAKE) moyen   P(FAKE) maximum")
print("-" * 70)
print(f"Scénario A      {val_fake['score_A'].min():.4f}            {val_fake['score_A'].mean():.4f}          {val_fake['score_A'].max():.4f}")
print(f"Scénario B      {val_fake['score_B'].min():.4f}            {val_fake['score_B'].mean():.4f}          {val_fake['score_B'].max():.4f}")
print(f"Scénario C      {val_fake['score_C'].min():.4f}            {val_fake['score_C'].mean():.4f}          {val_fake['score_C'].max():.4f}")
print()

# BONUS : Afficher les coefficients du méta-learner
print("=" * 70)
print("COEFFICIENTS DU MÉTA-LEARNER")
print("=" * 70)
print(f"β_Meso4   : {meta_model.coef_[0][0]:.4f}")
print(f"β_Xception: {meta_model.coef_[0][1]:.4f}")
print(f"β_UCF     : {meta_model.coef_[0][2]:.4f}")
print(f"β_F3Net   : {meta_model.coef_[0][3]:.4f}")

In [ ]:
COLORS_INDIV = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
COLORS_ENS   = ['#9467bd', '#8c564b', '#e377c2']

# ────────────────────────────────────────────────────────────────
# Figure 1 — Courbes ROC (Test Set)
# ────────────────────────────────────────────────────────────────
fig1, ax = plt.subplots(figsize=(8, 7))

# Modèles individuels
for model, color in zip(MODEL_NAMES, COLORS_INDIV):
    fpr, tpr, _ = roc_curve(y_test, RESULTS[model]['scores_test'])
    auc_v = RESULTS[model]['test']['AUC']
    ax.plot(fpr, tpr, color=color, lw=2,
            label=f'{model} (AUC = {auc_v:.3f})')

# Scénarios ensemble
ens_config = [
    ('Scén. A — Vote maj.',    'Scenario_A', COLORS_ENS[0], '--'),
    ('Scén. B — Moy. pond.',   'Scenario_B', COLORS_ENS[1], '-.'),
    ('Scén. C — Méta-Learn.',  'Scenario_C', COLORS_ENS[2], ':'),
]
for label_e, key, color, ls in ens_config:
    fpr, tpr, _ = roc_curve(y_test, RESULTS[key]['scores_test'])
    auc_v = RESULTS[key]['test']['AUC']
    ax.plot(fpr, tpr, color=color, lw=2.5, ls=ls,
            label=f'{label_e} (AUC = {auc_v:.3f})')

ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Aléatoire (AUC = 0.500)')
ax.set_xlabel('Taux de Faux Positifs (FPR)', fontsize=12)
ax.set_ylabel('Taux de Vrais Positifs (TPR)', fontsize=12)
ax.set_title(
    'Courbes ROC — Test Set\n'
    '(Modèles individuels [poids gelés FF++ c23] vs Scénarios Ensemble)',
    fontsize=12
)
ax.legend(loc='lower right', fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()

roc_path = f'{FIGS_DIR}/roc_curves_test.png'
fig1.savefig(roc_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ Figure 1 sauvegardée : {roc_path}')

# ────────────────────────────────────────────────────────────────
# Figure 2 — Matrices de Confusion (Test Set, seuil 0.5)
# ────────────────────────────────────────────────────────────────
cm_items = [
    ('Meso4',              (test_df['P_Meso4'].values       >= 0.5).astype(int)),
    ('XceptionNet',        (test_df['P_XceptionNet'].values >= 0.5).astype(int)),
    ('UCF',                (test_df['P_UCF'].values         >= 0.5).astype(int)),
    ('F3Net',              (test_df['P_F3Net'].values        >= 0.5).astype(int)),
    ('Scén. A — Vote maj.',   RESULTS['Scenario_A']['preds_test']),
    ('Scén. B — Moy. pond.',  (RESULTS['Scenario_B']['scores_test'] >= 0.5).astype(int)),
    ('Scén. C — Méta-Learn.', (RESULTS['Scenario_C']['scores_test'] >= 0.5).astype(int)),
]

fig2, axes = plt.subplots(2, 4, figsize=(18, 9))
axes_flat  = axes.flatten()

for idx, (title, preds) in enumerate(cm_items):
    cm  = confusion_matrix(y_test, preds)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['REAL', 'FAKE'])
    disp.plot(ax=axes_flat[idx], colorbar=False, cmap='Blues')
    acc_v = accuracy_score(y_test, preds)
    f1_v  = f1_score(y_test, preds, zero_division=0)
    axes_flat[idx].set_title(f'{title}\nAcc={acc_v:.3f}  F1={f1_v:.3f}', fontsize=9)

# Masquer le 8e subplot (vide)
axes_flat[7].axis('off')

fig2.suptitle(
    'Matrices de Confusion — Test Set (seuil = 0.5)\n'
    'Ligne 1 : Modèles individuels  │  Ligne 2 : Scénarios Ensemble',
    fontsize=12, y=1.01
)
plt.tight_layout()

cm_path = f'{FIGS_DIR}/confusion_matrices_test.png'
fig2.savefig(cm_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ Figure 2 sauvegardée : {cm_path}')


## Comparative Summary

In [ ]:
ENTRY_ORDER = [
    ('Meso4',       'Individuel'),
    ('XceptionNet', 'Individuel'),
    ('UCF',         'Individuel'),
    ('F3Net',       'Individuel'),
    ('Scenario_A',  'Ensemble'),
    ('Scenario_B',  'Ensemble'),
    ('Scenario_C',  'Ensemble'),
]

DISPLAY_NAMES = {
    'Meso4'      : 'Meso4',
    'XceptionNet': 'XceptionNet',
    'UCF'        : 'UCF',
    'F3Net'      : 'F3Net',
    'Scenario_A' : 'Scén. A — Vote maj.',
    'Scenario_B' : 'Scén. B — Moy. pond.',
    'Scenario_C' : 'Scén. C — Méta-Learn.',
}

rows = []
for key, cat in ENTRY_ORDER:
    for split in ['train', 'test']:
        m = RESULTS[key][split]
        rows.append({
            'Modèle / Scénario': DISPLAY_NAMES[key],
            'Catégorie'        : cat,
            'Split'            : split.capitalize(),
            'AUC'              : round(m['AUC'],      4),
            'Accuracy'         : round(m['Accuracy'], 4),
            'F1'               : round(m['F1'],       4),
            'EER'              : round(m['EER'],      4),
        })

summary_df = pd.DataFrame(rows)

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)
pd.set_option('display.float_format', '{:.4f}'.format)

print('=' * 95)
print('TABLEAU COMPARATIF SYNTHÉTIQUE — Métriques Train + Test')
print('Modèles entraînés sur FF++ c23 (poids gelés), évalués sur images de diffusion')
print('=' * 95)
print(summary_df.to_string(index=False))
print('=' * 95)

# Sauvegarde CSV
summary_path = f'{RESULTS_DIR}/metrics_summary.csv'
summary_df.to_csv(summary_path, index=False)
print(f'\n✅ Tableau sauvegardé : {summary_path}')


## Save & Verification

In [ ]:
print('Sauvegarde des fichiers de scores ensemble ...')
print()

META_COLS = ['filepath', 'label', 'method', 'split']

for split_name, df in [('train', train_df), ('test', test_df)]:
    # Colonnes métadonnées disponibles (tolérance si certaines absentes)
    available_meta = [c for c in META_COLS if c in df.columns]
    out = df[available_meta + PROB_COLS].copy()

    sk = 'scores_train' if split_name == 'train' else 'scores_test'
    out['score_A'] = RESULTS['Scenario_A'][sk]
    out['score_B'] = RESULTS['Scenario_B'][sk]
    out['score_C'] = RESULTS['Scenario_C'][sk]

    out_path = f'{RESULTS_DIR}/ensemble_scores_{split_name}.csv'
    out.to_csv(out_path, index=False)

    print(f'  ✅ {split_name} → {os.path.basename(out_path)}')
    print(f'     Lignes   : {len(out):,}')
    print(f'     Colonnes : {list(out.columns)}')
    print()

# ── Vérification finale du coffre-fort ───────────────────────────
print('━' * 65)
forbidden_files = [
    f'{RESULTS_DIR}/val_probs.csv',
    f'{RESULTS_DIR}/ensemble_scores_val.csv',
    f'{SPLITS_DIR}/val_manifest.csv',   # ne pas avoir été lu en lecture
]
vault_ok = True
for fp in forbidden_files[:2]:          # seuls les résultats val sont interdits
    if os.path.isfile(fp):
        vault_ok = False
        print(f'  ⚠️  COFFRE-FORT COMPROMIS : {os.path.basename(fp)} existe')

print(f'  Coffre-fort val : {"✅ INTACT" if vault_ok else "⚠️  COMPROMIS"}')
print('━' * 65)

print()
print('=' * 65)
print('  ✅ NOTEBOOK 05 TERMINÉ AVEC SUCCÈS')
print()
print('  Fichiers générés dans data/results/ :')
print('    → metrics_summary.csv')
print('    → ensemble_scores_train.csv')
print('    → ensemble_scores_test.csv')
print('    → figures/roc_curves_test.png')
print('    → figures/confusion_matrices_test.png')
print()
print('  Prochaine étape : Notebook 06 — Ouverture coffre-fort validation')
print('  (Fenêtre autorisée : 13–16 avril 2026 — MAINTENANT DISPONIBLE)')
print('=' * 65)
